### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

* Long-running conversations that exceed context windows.
* Multi-turn dialogues with extensive history.
* Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model


# Initialize Groq model
model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

# Message Based Summarization
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,                  # Pass model object here
            trigger=("messages", 10),     # summarize after 10 messages
            keep=("messages", 4)          # keep the 4 most recent messages
        )
    ]
)

In [6]:
### Run with thread id
config = {"configurable": {"thread_id": "test-1"}}   # this will uniquely identify my user

In [10]:
from langchain_core.messages import HumanMessage
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4+4?",
]

for q in questions:
    response = agent.invoke({
        "messages": [
            HumanMessage(content=q)
        ]
    } , config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='50229973-ab6e-4786-b0c7-c33fe49f87f5'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e68f5-f6d4-7e32-843b-a6b65acd46e1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 28, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='50229973-ab6e-4786-b0c7-c33fe49f87f5'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e68f5-f6d4-7e32-843b-a6b65acd46e1-0', tool_c

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 41.970385767s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fre

### Token Size

In [ ]:
from langchain_core import tools
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""

    return f""" Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free WiFi"""

model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens" , 550),
            keep = ("tokens" , 200)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}} 

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars = 1 token


In [13]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])

    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(response["messages"])
    print("-" * 50)

Paris: ~146 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='db133f9d-2d65-437a-a228-673a0850895d'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking to find hotels in Paris. Let me check the available tools. There's a function called search_hotels that takes a city parameter. The description says it returns a long response to use more tokens. Since the user wants hotels in Paris, I need to call this function with the city set to Paris. I'll make sure the parameters are correctly formatted as JSON within the tool_call tags. No other functions are available, so this should be the only tool call needed.\n", 'tool_calls': [{'id': 'x11r35z6d', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 155, 'total_tokens': 280, 'completion_time': 0.203030927, 'completion_tokens

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01ksaxjvxkedjvhfdpr73vhrrb` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4355, Requested 1715. Please try again in 700ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

### Human-in-the-loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [5]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [6]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7b5e11bf-4073-48f9-ab76-0cbedd1b9773'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants me to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool function. The parameters required are recipient, subject, and body. All three are required. The recipient is john@test.com, subject is Hello, and body is How are you? I need to structure the JSON arguments correctly. Let me make sure there are no typos. The function name is send_email_tool. So the tool_call should include the name and the arguments with those parameters. I'll double-check the email address and the message content. Everything looks good. Time to format the response as specified.\n", 'tool_calls': [{'id': 'a8rbsnb5n', 'function': {'arguments': '{"

In [7]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject "Hello". Let me know if you need anything else!


In [8]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='7b5e11bf-4073-48f9-ab76-0cbedd1b9773'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants me to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool function. The parameters required are recipient, subject, and body. All three are required. The recipient is john@test.com, subject is Hello, and body is How are you? I need to structure the JSON arguments correctly. Let me make sure there are no typos. The function name is send_email_tool. So the tool_call should include the name and the arguments with those parameters. I'll double-check the email address and the message content. Everything looks good. Time to format the response as specified.\n", 'tool_calls': [{'id': 'a8rbsnb5n', 'function': {'arguments': '{"

### Reject

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [10]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [11]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It looks like the email to john@test.com couldn't be sent. Would you like me to:
1. Try sending it again?
2. Check the email content/description?
3. Help with a different request?

Let me know how you'd like to proceed!


In [12]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ec9e053a-d717-4044-bb56-d98caeb1d24b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools.\n\nThere's a send_email_tool function. It requires recipient, subject, and body. The parameters are all provided in the user's request. So I need to call send_email_tool with those arguments. No need to use read_email_tool here since the user isn't asking to read an email. Just confirm that all required parameters are present and then structure the tool call accordingly.\n", 'tool_calls': [{'id': 'p2etpxvfc', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': 

### Editing

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [14]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [15]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='f629e93e-788e-489b-a426-b4ce64e53a58'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. The send_email_tool is the right one here because the user is asking to send an email. The parameters required for send_email_tool are recipient, subject, and body. The user provided all three: recipient is wrong@email.com, subject is 'Test', and body is 'Hello'. I should make sure all required parameters are included. Yep, looks like everything's there. So I'll call the send_email_tool with those arguments.\n", 'tool_calls': [{'id': 'k7xnt5bj5', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subjec

In [16]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email has been successfully sent to **correct@email.com** with the subject **'Corrected Subject'**. Let me know if you need further assistance!


In [17]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='f629e93e-788e-489b-a426-b4ce64e53a58'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. The send_email_tool is the right one here because the user is asking to send an email. The parameters required for send_email_tool are recipient, subject, and body. The user provided all three: recipient is wrong@email.com, subject is 'Test', and body is 'Hello'. I should make sure all required parameters are included. Yep, looks like everything's there. So I'll call the send_email_tool with those arguments.\n", 'tool_calls': [{'id': 'k7xnt5bj5', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subjec